# Memoria 解説生成ノートブック
**モデル:** `gemini-2.5-pro-preview-05-06`

## 手順
1. セル1を実行してライブラリをインストール
2. セル2でAPIキーを設定
3. セル3でquestions.jsonをアップロード
4. セル4で解説生成を開始（約1〜2時間かかります）
5. セル5で結果をダウンロード

In [ ]:
# セル1: ライブラリインストール
!pip install -q google-generativeai

import google.generativeai as genai
import json
import time
import csv
from pathlib import Path
from google.colab import files

print("インストール完了")

In [ ]:
# セル2: APIキー設定
# ↓ ここにAI StudioのAPIキーを貼り付け
GEMINI_API_KEY = "YOUR_API_KEY_HERE"

genai.configure(api_key=GEMINI_API_KEY)

# 利用可能なモデルを確認
print("利用可能なモデル:")
for m in genai.list_models():
    if 'generateContent' in [method.name for method in m.supported_generation_methods]:
        if 'gemini-2' in m.name:
            print(f"  {m.name}")

# モデル初期化
model = genai.GenerativeModel('gemini-2.5-pro-preview-05-06')

# テスト
response = model.generate_content("1+1は？")
print(f"\nテスト成功: {response.text.strip()[:50]}")

In [ ]:
# セル3: questions.json アップロード
print("questions.json をアップロードしてください")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
with open(filename, 'r', encoding='utf-8') as f:
    questions = json.load(f)
print(f"読み込み完了: {len(questions)}問")

# カテゴリ別の問題数を表示
from collections import Counter
cats = Counter(q['category'] for q in questions)
for cat, count in cats.most_common():
    print(f"  {cat}: {count}問")

In [ ]:
# セル4: 解説生成（メイン処理）

CHOICE_LABELS = ['A', 'B', 'C', 'D', 'E']
DELAY_BETWEEN = 1.5      # 各問題の間隔（秒）
SAVE_INTERVAL = 50       # 何問ごとに中間保存するか
results_file = "explanations_progress.json"


def build_prompt(q):
    """1問分の解説生成プロンプト"""
    choices_text = ""
    for i, choice in enumerate(q['choices']):
        label = CHOICE_LABELS[i]
        choices_text += f"  {label}. {choice}\n"
    correct = ', '.join(q['correct_answer'])
    return f"""あなたは看護師国家試験の教育専門家です。
以下の問題について、正答の根拠を中心に簡潔で正確な解説を作成してください。

【問題】{q['question_text']}
【選択肢】
{choices_text}
【正答】{correct}

解説のルール:
- 200〜300字程度で簡潔にまとめる
- なぜ正答が正しいのか、根拠を明確に述べる
- 重要な誤答選択肢についても簡単に触れる
- 医学的に正確な内容にする
- 学生が理解しやすい平易な表現を使う
- 解説のみを出力し、「解説:」などの接頭辞は不要"""


def generate_explanation(q, max_retries=3):
    """1問の解説を生成（リトライ付き）"""
    prompt = build_prompt(q)
    for attempt in range(max_retries):
        try:
            response = model.generate_content(prompt)
            text = response.text.strip()
            if len(text) > 20:
                return text
            else:
                print(f"  短すぎる応答、リトライ ({attempt+1}/{max_retries})")
        except Exception as e:
            error_msg = str(e)
            if '429' in error_msg or 'quota' in error_msg.lower() or 'resource' in error_msg.lower():
                wait = 30 * (attempt + 1)
                print(f"  レート制限、{wait}秒待機...")
                time.sleep(wait)
            else:
                print(f"  エラー: {error_msg[:100]}")
                if attempt < max_retries - 1:
                    time.sleep(5)
    return ""


# 中断再開: 既存の結果があれば読み込み
if Path(results_file).exists():
    with open(results_file, 'r', encoding='utf-8') as f:
        results = json.load(f)
    print(f"中断データ読み込み: {len(results)}問生成済み")
else:
    results = {}

total = len(questions)
success_count = sum(1 for v in results.values() if v)
error_count = 0

print(f"\n===== 解説生成開始 =====")
print(f"総問題数: {total}")
print(f"生成済み: {success_count}")
print(f"残り: {total - success_count}")
print(f"予想時間: 約{((total - success_count) * DELAY_BETWEEN) // 60}分")
print(f"========================\n")

start_time = time.time()

for i, q in enumerate(questions):
    qid = q['question_id']

    # 既に生成済みならスキップ
    if qid in results and results[qid]:
        continue

    # 生成
    explanation = generate_explanation(q)

    if explanation:
        results[qid] = explanation
        success_count += 1
    else:
        results[qid] = ""
        error_count += 1

    # 進捗表示
    done = i + 1
    if done % 10 == 0 or done == total:
        elapsed = time.time() - start_time
        remaining = total - done
        eta_min = (elapsed / max(done - len([v for v in results.values() if v]) + success_count, 1)) * remaining / 60
        pct = round(done / total * 100, 1)
        print(f"[{done}/{total}] {pct}%  成功:{success_count} エラー:{error_count}  残り約{eta_min:.0f}分")

    # 中間保存
    if done % SAVE_INTERVAL == 0:
        with open(results_file, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        print(f"  → 中間保存完了")

    # レート制限回避
    time.sleep(DELAY_BETWEEN)

# 最終保存
with open(results_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

elapsed_total = (time.time() - start_time) / 60
print(f"\n===== 生成完了 =====")
print(f"成功: {success_count}")
print(f"エラー: {error_count}")
print(f"所要時間: {elapsed_total:.1f}分")
print(f"====================")

In [ ]:
# セル5: 結果統合 & ダウンロード

# 解説をquestionsに統合
for q in questions:
    qid = q['question_id']
    if qid in results and results[qid]:
        q['explanation'] = results[qid]

# 確認
with_exp = sum(1 for q in questions if q.get('explanation') and len(q['explanation']) > 20)
print(f"解説付き: {with_exp}/{len(questions)}問")

# サンプル表示
for q in questions[:3]:
    print(f"\n--- {q['question_id']} ---")
    print(f"問題: {q['question_text'][:60]}...")
    exp = q.get('explanation', '(なし)')
    print(f"解説: {exp[:100]}...")

# JSON保存
output_json = "questions_with_gemini_explanations.json"
with open(output_json, 'w', encoding='utf-8') as f:
    json.dump(questions, f, ensure_ascii=False, indent=2)
print(f"\nJSON保存: {output_json}")

# CSV保存（Google Sheetsインポート用）
output_csv = "questions_with_gemini_explanations.csv"
with open(output_csv, 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerow([
        'question_id', 'department', 'exam_year', 'exam_number',
        'category', 'subcategory', 'subtopic', 'difficulty',
        'question_text',
        'choice_a', 'choice_b', 'choice_c', 'choice_d', 'choice_e',
        'correct_answer', 'explanation',
        'has_image', 'image_url', 'is_multi_select', 'source', 'created_at'
    ])
    for q in questions:
        choices = q.get('choices', [])
        writer.writerow([
            q.get('question_id', ''),
            q.get('department', ''),
            q.get('exam_year', ''),
            q.get('exam_number', ''),
            q.get('category', ''),
            q.get('subcategory', ''),
            q.get('subtopic', ''),
            q.get('difficulty', ''),
            q.get('question_text', ''),
            choices[0] if len(choices) > 0 else '',
            choices[1] if len(choices) > 1 else '',
            choices[2] if len(choices) > 2 else '',
            choices[3] if len(choices) > 3 else '',
            choices[4] if len(choices) > 4 else '',
            ','.join(q.get('correct_answer', [])),
            q.get('explanation', ''),
            q.get('has_image', False),
            q.get('image_url', ''),
            q.get('is_multi_select', False),
            q.get('source', ''),
            q.get('created_at', ''),
        ])
print(f"CSV保存: {output_csv}")

# ダウンロード
print("\nダウンロード開始...")
files.download(output_json)
files.download(output_csv)
print("完了！")

In [ ]:
# セル6（オプション）: エラーだった問題だけ再生成

failed = [q for q in questions if not q.get('explanation') or len(q.get('explanation', '')) < 20]
print(f"解説未生成: {len(failed)}問")

if len(failed) > 0:
    print("再生成を開始します...")
    for i, q in enumerate(failed):
        qid = q['question_id']
        explanation = generate_explanation(q)
        if explanation:
            q['explanation'] = explanation
            results[qid] = explanation
            print(f"  [{i+1}/{len(failed)}] {qid} → 成功")
        else:
            print(f"  [{i+1}/{len(failed)}] {qid} → 失敗")
        time.sleep(DELAY_BETWEEN)

    # 再保存
    with open(results_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print("再生成完了。セル5を再実行してダウンロードしてください。")
else:
    print("全問題の解説が生成済みです！")